# Grant and Scholarship Application Evaluator demo: multi agent trace
Requires LLM_API_KEY in the .env file at the project root. Run the cells in order.

In [ ]:
import json, os, sys
sys.path.insert(0, os.path.abspath('..'))
from scholardesk import ScholarDesk, config

assert config.configured() or print('Set LLM_API_KEY in .env first'), 'no API key'
desk = ScholarDesk()
samples = {s['label']: s['text'] for s in json.load(open('../data/samples.json'))}

async def evaluate(label):
    state = desk.submit(samples[label])
    await desk.wait_idle()
    return state

def trace(state):
    events = desk.bus.history[state.app_id]
    t0 = events[0]['ts']
    for e in events:
        d = e['data']
        detail = d.get('tool') or d.get('text') or d.get('error') or ''
        if e['type'] == 'tool_result': detail = f"{d['tool']} -> {d['result'][:90]}"
        print(f"{e['ts'] - t0:6.2f}s  {e['agent'] or '':<18} {e['type']:<14} {str(detail)[:110]}")

def marksheet(state):
    s = state.snapshot()
    for r in s['rows']:
        print(f"{r['line_id']}  {r['criterion']:<22} score {r['score']}  weight {r['weight']:.0%}  points {r['points']}  | {r['agent']}: {r['reasoning']}")
    print('decision', s['decision']['decision'], '| total', s['total'], '| cited', s['decision']['cited'])

Scenario 1. A strong application: three workers run concurrently, the manager questions one, then decides.

In [ ]:
s1 = await evaluate('Strong merit, real need')
trace(s1)
print()
marksheet(s1)

Scenario 2. A hard rule failure: rejected outright with the failed rules named.

In [ ]:
s2 = await evaluate('Fails score and income rules')
trace(s2)
print(s2.eligibility)

Scenario 3. Honest failure: income is not stated, so agents report it missing and the application is held.

In [ ]:
s3 = await evaluate('Income not stated')
trace(s3)
print(s3.missing, s3.decision)

Scenario 4. The borderline pair, then the Consistency Agent audits them.

In [ ]:
a = await evaluate('Borderline A')
b = await evaluate('Borderline B')
print(a.applicant, a.decision['decision'], a.snapshot()['total'])
print(b.applicant, b.decision['decision'], b.snapshot()['total'])
job = desk.submit_consistency(a.app_id, b.app_id)
await desk.wait_idle()
print(job['result']['consistent'], job['result']['explanation'])

Append only run log. Every line carries the hash of the previous line.

In [ ]:
rows = desk.log.rows()
print(len(rows), 'entries; chain valid:', desk.log.verify()[0])
for r in rows[:4]: print(r['seq'], r['agent'], r['event'], r['prev'][:10], '->', r['hash'][:10])